# Chapter 3 — Raters, judges, PPI and generalizability

Companion notebook for *Evaluating Large Language Models in High-Stakes Domains* (Costa, 2026). All data in this notebook are simulated; the generating process is in the code and described in the text.

In [ ]:
"""Figures and worked-example numbers for 'Evaluating Large Language Models in High-Stakes Domains'.
All data are analytic or simulated; generating processes are stated in the text.
Run:  python figures.py   (writes PNGs and results.json into ./figures)
"""
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, minimize_scalar

OUT = os.path.join(os.getcwd(), "figures_out")
os.makedirs(OUT, exist_ok=True)

# palette (categorical, fixed order) and ink
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
    "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.dpi": 200, "savefig.dpi": 200,
    "savefig.facecolor": "white",
})
results = {}
se = lambda x: x.std(ddof=1)/np.sqrt(len(x))
def wilson(x, n, z=1.96):
    p = x/n; c = (p + z*z/(2*n))/(1 + z*z/n)
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/(1 + z*z/n)
    return c-h, c+h
def clopper_upper(x, n, conf=0.95):
    return 1.0 if x == n else stats.beta.ppf(conf, x+1, n-x)


## Worked example, Ch. 3

In [ ]:
rng = np.random.default_rng(33)
N3 = 3000                                   # outputs scored by the judge
q = rng.beta(5, 2, N3)                      # latent quality of each output
s_true = np.clip(q + rng.normal(0, 0.05, N3), 0, 1)   # expert ("true") rubric score
# (a) three human raters on a random subset of 150 outputs: s_ir = s_i + b_r + e_ir
n_r, R = 150, 3
sub = rng.choice(N3, n_r, replace=False)
b_r = np.array([0.0, 0.06, -0.03])
ratings = s_true[sub][:, None] + b_r[None, :] + rng.normal(0, 0.10, (n_r, R))
ratings = np.clip(ratings, 0, 1)
def kripp_alpha_interval(M):             # M: items x raters, no missing
    n_i, R_ = M.shape
    Do = np.mean([np.sum((M[i][:, None]-M[i][None, :])**2)/(R_-1) for i in range(n_i)])/R_
    pooled = M.ravel(); n_p = len(pooled)
    De = np.sum((pooled[:, None]-pooled[None, :])**2)/(n_p*(n_p-1))
    return 1 - Do/De
alpha_hat = kripp_alpha_interval(ratings)
bootA = np.array([kripp_alpha_interval(ratings[rng.integers(0, n_r, n_r)]) for _ in range(2000)])
# generalizability theory: two-way random effects (items x raters), method of moments
grand = ratings.mean(); mi = ratings.mean(1); mr = ratings.mean(0)
SS_i = R*np.sum((mi-grand)**2); SS_r = n_r*np.sum((mr-grand)**2)
SS_e = np.sum((ratings - mi[:, None] - mr[None, :] + grand)**2)
MS_i, MS_r, MS_e = SS_i/(n_r-1), SS_r/(R-1), SS_e/((n_r-1)*(R-1))
var_e = MS_e; var_i = max((MS_i-MS_e)/R, 0); var_r = max((MS_r-MS_e)/n_r, 0)
se_items_only = np.sqrt(var_i/n_r + var_e/(n_r*R))        # what the Ch.2 SE across item means gives
se_full = np.sqrt(var_i/n_r + var_r/R + var_e/(n_r*R))    # adds the rater facet
se_full_R5 = np.sqrt(var_i/n_r + var_r/5 + var_e/(n_r*5))
se_full_n300 = np.sqrt(var_i/300 + var_r/R + var_e/(300*R))
results["ch3"] = {
    "ALPHA": f"{alpha_hat:.3f}", "ALPHA_CI": f"[{np.percentile(bootA, 2.5):.3f}, {np.percentile(bootA, 97.5):.3f}]",
    "VAR_I": f"{var_i:.4f}", "VAR_R": f"{var_r:.4f}", "VAR_E": f"{var_e:.4f}",
    "SE_ITEMS": f"{se_items_only:.4f}", "SE_FULL": f"{se_full:.4f}",
    "SE_R5": f"{se_full_R5:.4f}", "SE_N300": f"{se_full_n300:.4f}",
    "RATER_SD": f"{np.sqrt(var_r):.3f}", "PCT_UNDER": f"{100*(1-se_items_only/se_full):.0f}",
}
# (b) judge validation for the unsafe category
p_unsafe = 1/(1+np.exp(-(-2.6 - 3.0*(q-0.6))))         # unsafe more likely for low-quality outputs
unsafe = rng.binomial(1, p_unsafe)                       # expert label
sens_true, spec_true = 0.60, 0.97
judge_unsafe = np.where(unsafe == 1, rng.binomial(1, sens_true, N3), rng.binomial(1, 1-spec_true, N3))
rand100 = rng.choice(N3, 100, replace=False)
u_r = unsafe[rand100]; j_r = judge_unsafe[rand100]
tp_r = int(((u_r==1)&(j_r==1)).sum()); pos_r = int((u_r==1).sum())
enr = np.concatenate([rng.choice(np.where(unsafe==1)[0], 50, replace=False),
                      rng.choice(np.where(unsafe==0)[0], 50, replace=False)])
u_e = unsafe[enr]; j_e = judge_unsafe[enr]
tp_e = int(((u_e==1)&(j_e==1)).sum()); pos_e = int((u_e==1).sum())
tn_e = int(((u_e==0)&(j_e==0)).sum()); neg_e = int((u_e==0).sum())
# agreement on the random sample vs sensitivity
agree_r = float((u_r == j_r).mean())
lo_r, hi_r = wilson(tp_r, pos_r) if pos_r else (np.nan, np.nan)
lo_e, hi_e = wilson(tp_e, pos_e); lo_s, hi_s = wilson(tn_e, neg_e)
results["ch3"].update({
    "UNSAFE_RATE": f"{100*unsafe.mean():.1f}", "AGREE_R": f"{100*agree_r:.0f}",
    "POS_R": str(pos_r), "SENS_R": f"{tp_r/pos_r:.2f} [{lo_r:.2f}, {hi_r:.2f}]" if pos_r else "n/a",
    "SENS_E": f"{tp_e/pos_e:.2f} [{lo_e:.2f}, {hi_e:.2f}]", "SPEC_E": f"{tn_e/neg_e:.2f} [{lo_s:.2f}, {hi_s:.2f}]",
})
# (c) prediction-powered inference with a lenient judge
judge_score = np.clip(s_true + 0.06 + rng.normal(0, 0.15, N3), 0, 1)
lab = rng.choice(N3, 300, replace=False); unl = np.setdiff1d(np.arange(N3), lab)
s_lab, sh_lab, sh_unl = s_true[lab], judge_score[lab], judge_score[unl]
n_l, N_u = len(s_lab), len(sh_unl)
v_lab = np.var(sh_lab, ddof=1); c = np.cov(s_lab, sh_lab, ddof=1)[0, 1]
lam = float(np.clip(c/(v_lab*(1 + n_l/N_u)), 0, 1))
est_pp = lam*sh_unl.mean() + (s_lab - lam*sh_lab).mean()
var_pp = lam**2*np.var(sh_unl, ddof=1)/N_u + np.var(s_lab - lam*sh_lab, ddof=1)/n_l
half_pp = 1.96*np.sqrt(var_pp); half_cl = 1.96*se(s_lab)
results["ch3"].update({
    "TRUE_MEAN": f"{s_true.mean():.3f}", "SD_S": f"{s_true.std(ddof=1):.2f}",
    "JUDGE_MEAN": f"{judge_score.mean():.3f}", "JUDGE_BIAS": f"{(judge_score-s_true).mean():+.3f}",
    "ERR_SD": f"{(judge_score-s_true).std(ddof=1):.2f}",
    "CLASSIC": f"{s_lab.mean():.3f} ± {half_cl:.3f}", "PPI": f"{est_pp:.3f} ± {half_pp:.3f}",
    "LAMBDA": f"{lam:.2f}", "PPI_RED": f"{100*(1-half_pp/half_cl):.0f}",
})

In [ ]:
import json
print(json.dumps(results, indent=2, ensure_ascii=False))